# Mudra Classification with SVM

This notebook implements an SVM-based classification model for Indian Classical Mudras using the 'infused' dataset.

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from skimage.feature import hog
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

## 1. Load Data and Extract Features

We will resize the images and extract HOG (Histogram of Oriented Gradients) features.

In [ ]:
def load_mudra_data(data_dir, img_size=(128, 128)):
    features = []
    labels = []
    class_names = []
    
    if not os.path.exists(data_dir) or not os.listdir(data_dir):
        print(f"Warning: Data directory '{data_dir}' is empty or does not exist.")
        return None, None, None

    for class_folder in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_folder)
        if os.path.isdir(class_path):
            class_names.append(class_folder)
            print(f"Loading class: {class_folder}")
            for img_file in os.listdir(class_path):
                img_path = os.path.join(class_path, img_file)
                try:
                    # Read and resize
                    img = cv2.imread(img_path)
                    if img is None: continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    img = cv2.resize(img, img_size)
                    
                    # Extract HOG features
                    # orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2) are standard
                    # These can be tuned for better performance
                    fd = hog(img, orientations=9, pixels_per_cell=(8, 8),
                             cells_per_block=(2, 2), visualize=False)
                    
                    features.append(fd)
                    labels.append(len(class_names) - 1)
                except Exception as e:
                    print(f"  Error loading {img_file}: {e}")
                    
    return np.array(features), np.array(labels), class_names

DATA_PATH = "data"
X, y, class_names = load_mudra_data(DATA_PATH)

if X is not None:
    print(f"\nTotal images loaded: {len(X)}")
    print(f"Feature vector shape: {X.shape[1]}")

## 2. Train-Test Split

In [ ]:
if X is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Training set: {X_train.shape[0]}, Test set: {X_test.shape[0]}")

## 3. Train SVM Model

We use an RBF kernel for non-linear classification.

In [ ]:
if X is not None:
    clf = svm.SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
    print("Training SVM model...")
    clf.fit(X_train, y_train)
    print("Model training complete.")

## 4. Evaluation

In [ ]:
if X is not None:
    y_pred = clf.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=class_names))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()